# 02 — Generar perfiles de voz con XTTS-v2

Este notebook convierte textos en audio usando perfiles de voz. No entrena un modelo. XTTS-v2 extrae las características de cada audio y las guarda para reutilizarlas.

Flujo: cargar el modelo, leer los audios, generar los perfiles y probar una síntesis.

## 1. Preparación del entorno y comprobación de CUDA

Se detecta si hay GPU CUDA. En CPU también funciona, pero es más lento. Las dependencias están en `requirements.txt`.

In [1]:
import os
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE)
print('CUDA disponible:', torch.cuda.is_available())


Dispositivo: cuda
CUDA disponible: True


## 2. Cargar el modelo XTTS-v2

Se carga el modelo multilingüe XTTS-v2. La primera ejecución puede descargarlo y tardar varios minutos. Luego se reutiliza la copia local. El modelo convierte los textos del asistente en audio.

La celda también configura la carga del checkpoint requerida por PyTorch.

In [2]:
import torch
from TTS.tts.configs.xtts_config import XttsConfig
torch.serialization.add_safe_globals([XttsConfig])
os.environ["COQUI_TOS_AGREED"] = "1"

# El checkpoint oficial de XTTS es confiable y requiere carga pickle completa.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

from TTS.api import TTS

tts = TTS(
    "tts_models/multilingual/multi-dataset/xtts_v2",
    gpu=torch.cuda.is_available(),
)
model = tts.synthesizer.tts_model

print("XTTS cargado")

/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.speakers = torch.load(speaker_file_path)
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/utils/io.py:54: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location, **kwargs)


XTTS cargado


## 3. Seleccionar la identidad y la grabación de referencia

Cada archivo WAV de `data/voice_samples/` representa una voz. Su nombre se usa como `VOICE_ID`: `winz.wav` genera `winz.pt`. En la celda 6, selecciona la voz en el menú desplegable.

Usa audios de 10 a 30 segundos, con poco ruido y una sola persona. La voz debe estar autorizada.

In [3]:
from pathlib import Path

# Encontrar la raíz aunque Jupyter se haya iniciado desde notebooks/.
PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "data" / "voice_samples").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró data/voice_samples. Inicia Jupyter desde la raíz del proyecto."
    )

VOICE_SAMPLES_DIR = PROJECT_DIR / "data" / "voice_samples"
VOICE_PROFILES_DIR = PROJECT_DIR / "data" / "voice_profiles"
VOICE_PROFILES_DIR.mkdir(parents=True, exist_ok=True)
voice_sample_paths = sorted(VOICE_SAMPLES_DIR.glob("*.wav"))
if not voice_sample_paths:
    raise FileNotFoundError(f"No hay archivos WAV en {VOICE_SAMPLES_DIR.resolve()}")

available_voice_ids = {path.stem for path in voice_sample_paths}
import ipywidgets as widgets
from IPython.display import display

# Selecciona la voz que se usará para generar el audio de prueba.
voice_selector = widgets.Dropdown(
    options=sorted(available_voice_ids),
    value=sorted(available_voice_ids)[0],
    description="Voz:",
    style={"description_width": "initial"}
)
VOICE_ID = voice_selector.value

def select_voice(change):
    global VOICE_ID
    VOICE_ID = change["new"]
    print("Voz seleccionada para la síntesis:", VOICE_ID)

voice_selector.observe(select_voice, names="value")
display(voice_selector)
print("Raíz del proyecto:", PROJECT_DIR)
print(f"Audios encontrados: {len(voice_sample_paths)}")
print("Voz seleccionada para la síntesis:", VOICE_ID)
for path in voice_sample_paths:
    print(" -", path.stem, "→", path.name)

voice_selector

Dropdown(description='Voz:', options=('ana', 'diego', 'elsa', 'estuardo', 'nicole', 'winz'), style=Description…

Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz
Audios encontrados: 6
Voz seleccionada para la síntesis: ana
 - ana → ana.wav
 - diego → diego.wav
 - elsa → elsa.wav
 - estuardo → estuardo.wav
 - nicole → nicole.wav
 - winz → winz.wav


Dropdown(description='Voz:', options=('ana', 'diego', 'elsa', 'estuardo', 'nicole', 'winz'), style=Description…

## 4. Extraer las representaciones de voz

XTTS calcula `gpt_cond_latent` y `speaker_embedding`. El bucle procesa todos los WAV y guarda un perfil por voz. XTTS-v2 sigue siendo necesario para sintetizar el audio.

In [4]:
import torch
import torchaudio

print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("TorchAudio:", torchaudio.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Capacidad GPU:", torch.cuda.get_device_capability(0))
print("Arquitecturas de PyTorch:", torch.cuda.get_arch_list())

torch.arange(3, device="cuda")
print("Prueba básica de CUDA: OK")

PyTorch: 2.7.1+cu128
CUDA de PyTorch: 12.8
TorchAudio: 2.7.1+cu128
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Capacidad GPU: (12, 0)
Arquitecturas de PyTorch: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120', 'compute_120']
Prueba básica de CUDA: OK


In [5]:
generated_profiles = []
failed_profiles = []

for voice_sample_path in voice_sample_paths:
    voice_id = voice_sample_path.stem
    profile_path = VOICE_PROFILES_DIR / f"{voice_id}.pt"
    try:
        with torch.inference_mode():
            gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
                audio_path=[str(voice_sample_path)],
                gpt_cond_len=6,
                max_ref_length=30,
                sound_norm_refs=False
            )
        torch.save({
            "voice_id": voice_id,
            "gpt_cond_latent": gpt_cond_latent.detach().cpu(),
            "speaker_embedding": speaker_embedding.detach().cpu(),
            "model": "XTTS-v2",
            "language": "es"
        }, profile_path)
        generated_profiles.append(profile_path)
        print(f"✓ {voice_id}: {profile_path.name}")
    except Exception as exc:
        failed_profiles.append((voice_id, str(exc)))
        print(f"✗ {voice_id}: {exc}")

print(f"\nPerfiles generados: {len(generated_profiles)}/{len(voice_sample_paths)}")
if failed_profiles:
    print("Revisar:", [voice_id for voice_id, _ in failed_profiles])

✓ ana: ana.pt
✓ diego: diego.pt
✓ elsa: elsa.pt
✓ estuardo: estuardo.pt
✓ nicole: nicole.pt
✓ winz: winz.pt

Perfiles generados: 6/6


## 5. Persistir el artefacto generado

Cada perfil se guarda como `data/voice_profiles/<VOICE_ID>.pt`. Incluye la identidad, las representaciones de voz, el modelo y el idioma. El resultado muestra qué perfiles se generaron.

Los audios y perfiles contienen información de voz. Manténlos en un espacio privado.

In [6]:
print("Perfiles disponibles para el asistente:")
for profile_path in sorted(VOICE_PROFILES_DIR.glob("*.pt")):
    print(" -", profile_path.stem, "→", profile_path.name)

Perfiles disponibles para el asistente:
 - ana → ana.pt
 - diego → diego.pt
 - elsa → elsa.pt
 - estuardo → estuardo.pt
 - nicole → nicole.pt
 - winz → winz.pt


## 6. Validar la síntesis

La prueba carga un perfil y convierte un texto nuevo en audio. La calidad depende del ruido, la duración y la expresividad del audio de referencia.

In [8]:
profile_path = VOICE_PROFILES_DIR / f"{VOICE_ID}.pt"
profile = torch.load(profile_path, map_location=DEVICE, weights_only=True)
text = "Hola, quiero realizar compras en línea y necesito ayuda para encontrar los mejores productos."
with torch.inference_mode():
    result = model.inference(
        text=text, language="es",
        gpt_cond_latent=profile["gpt_cond_latent"].to(DEVICE),
        speaker_embedding=profile["speaker_embedding"].to(DEVICE),
        enable_text_splitting=True
    )
from IPython.display import Audio, display
display(Audio(result["wav"], rate=24000))